# 01 - Netflix-paper visible units on MovieLens

## 1. Objective

This notebook follows the representation in *Restricted Boltzmann Machines for Collaborative Filtering*.
The goal here is **not** to train the RBM yet.
The goal is to transform MovieLens user–movie ratings into the paper’s visible-layer encoding.
This notebook is the real-number / Netflix-paper baseline counterpart to the hyperbolic-number preparation notebook.

## 2. Paper logic: from ratings table to visible units

Suppose we have \(M\) movies, \(N\) users, and integer rating values from 1 to \(K\).
For one user who rated \(m\) movies, the observation is represented as a \(K \times m\) binary indicator matrix \(V\).
Each column corresponds to one rated movie by that user.
Each column is a softmax visible unit.
If the user rated movie \(i\) as rating \(k\), then \(v_i^k = 1\) and all other entries in that column are 0.
Missing ratings are not encoded as visible states; they simply do not contribute.

For a user who rated \(m\) movies,
\[
V \in \{0,1\}^{K \times m},
\]
with
\[
v_i^k =
\begin{cases}
1, & \text{if the user rated movie } i \text{ as } k,\\
0, & \text{otherwise.}
\end{cases}
\]

Each column of \(V\) is a one-hot rating vector for one rated movie, so the visible layer is a collection of softmax units rather than a single scalar rating vector.

## 3. Paper equations relevant for the visible layer

The following equations are **from the paper** and included here to anchor notation.

### Eq. (1) Visible conditional distribution

$$
p(v_i^k = 1 \mid h) =
\frac{\exp\left(b_i^k + \sum_{j=1}^{F} h_j W_{ij}^k\right)}
{\sum_{l=1}^{K} \exp\left(b_i^l + \sum_{j=1}^{F} h_j W_{ij}^l\right)}
$$

This equation defines the softmax probability that movie i takes rating level k given the hidden units h.

---

### Eq. (2) Hidden conditional distribution

$$
p(h_j = 1 \mid V) =
\sigma\left(
 b_j +
 \sum_{i=1}^{m}
 \sum_{k=1}^{K}
 v_i^k W_{ij}^k
\right)
$$

This equation gives the probability that hidden unit h_j is active given the visible matrix V.

---

### Eq. (4) Energy function

$$
E(V,h)=
-\sum_{i=1}^{m}
\sum_{j=1}^{F}
\sum_{k=1}^{K}
W_{ij}^k h_j v_i^k
+
\sum_{i=1}^{m}\log Z_i
-
\sum_{i=1}^{m}
\sum_{k=1}^{K}
v_i^k b_i^k
-
\sum_{j=1}^{F}
h_j b_j
$$

The normalization term is defined as:

$$
Z_i =
\sum_{l=1}^{K}
\exp\left(
 b_i^l +
 \sum_{j=1}^{F} h_j W_{ij}^l
\right)
$$

## 4. Mapping MovieLens ratings to the paper’s \(K\) categories

The Netflix paper assumes integer ratings from 1 to \(K\), with \(K = 5\).
MovieLens ratings are typically in 0.5 increments.
Therefore, for the Netflix-paper baseline notebook, we must define an explicit rating vocabulary before constructing visible units.

We use a 10-level ordinal vocabulary for MovieLens:

\[
\mathcal{R} = [0.5, 1.0, 1.5, \dots, 5.0]
\]
and define
\[
K = 10
\]

In the original Netflix paper, \(K = 5\) because ratings are integer-valued.
In this MovieLens adaptation, we preserve the same softmax visible-unit logic but replace the 5-level scale with a 10-level scale.
This keeps the representation faithful to the paper while respecting MovieLens granularity.

## 5. Load the prepared MovieLens subset

At this stage we only need a long-form ratings table with columns equivalent to userId, movieId, rating.

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np


def resolve_project_root() -> Path:
    root = Path.cwd().resolve()
    if root.name == "notebooks":
        root = root.parent
    return root


def load_ratings_table() -> tuple[pd.DataFrame, Path]:
    root = resolve_project_root()
    candidates = [
        root / "data" / "processed" / "train_ratings.csv",
        root / "data" / "processed" / "ratings.csv",
        root / "data" / "processed" / "rating.csv",
        root / "data" / "rating.csv",
        root / "data" / "ratings.csv",
    ]

    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            if {"userId", "movieId", "rating"}.issubset(df.columns):
                return df[["userId", "movieId", "rating"]].copy(), path

    data_dir = root / "data"
    search_dirs = [data_dir / "processed", data_dir]
    for directory in search_dirs:
        if not directory.exists():
            continue
        for path in sorted(directory.glob("*.csv")):
            try:
                df = pd.read_csv(path)
            except Exception:
                continue
            if {"userId", "movieId", "rating"}.issubset(df.columns):
                return df[["userId", "movieId", "rating"]].copy(), path

    raise FileNotFoundError("No ratings table found with columns userId, movieId, rating.")


ratings_df, ratings_path = load_ratings_table()
print(f"Using ratings table: {ratings_path}")
print("Ratings shape:", ratings_df.shape)
ratings_df.head()

Using ratings table: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/train_ratings.csv
Ratings shape: (16000210, 3)


,userId,movieId,rating
0,28507,1176,4.0
1,131160,1079,3.0
2,131160,47,5.0
3,131160,21,3.0
4,85252,45,3.0


## 6. Build a user-specific paper-style visible matrix

We now select one example user with enough ratings and construct the paper-style visible representation \(V \in \{0,1\}^{K \times m}\).

In [ ]:
rating_levels = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
K = len(rating_levels)


def build_visible_matrix_for_user(user_ratings_df: pd.DataFrame, rating_levels: list[float]):
    df = user_ratings_df[["movieId", "rating"]].copy()
    df = df.sort_values("movieId").reset_index(drop=True)

    ordered_movie_ids = df["movieId"].tolist()
    ordered_ratings = [round(float(r), 1) for r in df["rating"].tolist()]

    rating_to_index = {round(r, 1): idx for idx, r in enumerate(rating_levels)}
    K = len(rating_levels)
    m = len(df)

    V = np.zeros((K, m), dtype=int)
    for col, rating in enumerate(ordered_ratings):
        if rating not in rating_to_index:
            raise ValueError(f"Rating {rating} not in rating_levels vocabulary.")
        row = rating_to_index[rating]
        V[row, col] = 1

    return V, ordered_movie_ids, ordered_ratings, rating_to_index


user_counts = ratings_df.groupby("userId").size().reset_index(name="count")
eligible_users = user_counts[user_counts["count"] >= 20] #condition to have at least 20 ratings
eligible_users = eligible_users.sort_values(["count", "userId"], ascending=[False, True])

if eligible_users.empty:
    raise ValueError("No user found with at least 20 ratings.")

selected_user_id = int(eligible_users.iloc[0]["userId"]) #select the first user with at least 20 ratings
user_ratings = ratings_df[ratings_df["userId"] == selected_user_id].copy()

V, ordered_movie_ids, ordered_ratings, rating_to_index = build_visible_matrix_for_user(
    user_ratings, rating_levels
)

In [12]:
m = V.shape[1]
preview_n = min(10, m)

print("Selected user:", selected_user_id)
print("K (number of rating levels):", K)
print("m (number of rated movies):", m)
print("Shape of V:", V.shape)
print("First 10 ordered_movie_ids:", ordered_movie_ids[:preview_n])
print("First 10 ordered_ratings:", ordered_ratings[:preview_n])
print("rating_to_index:", rating_to_index)

print("\nFirst 10 columns of V as numpy array:\n", V[:, :preview_n])

visible_preview_df = pd.DataFrame(
    V[:, :preview_n],
    index=rating_levels,
    columns=ordered_movie_ids[:preview_n],
)
visible_preview_df

Selected user: 118205
K (number of rating levels): 10
m (number of rated movies): 7677
Shape of V: (10, 7677)
First 10 ordered_movie_ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
First 10 ordered_ratings: [4.0, 4.0, 3.0, 3.0, 3.0, 4.0, 3.0, 4.0, 3.0, 4.0]
rating_to_index: {0.5: 0, 1.0: 1, 1.5: 2, 2.0: 3, 2.5: 4, 3.0: 5, 3.5: 6, 4.0: 7, 4.5: 8, 5.0: 9}

First 10 columns of V as numpy array:
 [[0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 1 1 1 0 1 0 1 0]
 [0 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 0 1 0 1 0 1]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]]


,1,2,3,4,5,6,7,8,9,10
0.5,0,0,0,0,0,0,0,0,0,0
1.0,0,0,0,0,0,0,0,0,0,0
1.5,0,0,0,0,0,0,0,0,0,0
2.0,0,0,0,0,0,0,0,0,0,0
2.5,0,0,0,0,0,0,0,0,0,0
3.0,0,0,1,1,1,0,1,0,1,0
3.5,0,0,0,0,0,0,0,0,0,0
4.0,1,1,0,0,0,1,0,1,0,1
4.5,0,0,0,0,0,0,0,0,0,0
5.0,0,0,0,0,0,0,0,0,0,0


## 7. Show the encoding explicitly

We now surface the shapes, a few example columns, and a compact table view to make the one-hot encoding obvious.

In this encoding, one movie corresponds to one visible softmax unit, and one observed rating corresponds to one active state in that unit.
Unrated movies are not given a visible column in the standard user-specific RBM setup.

## Missing rating

In the paper’s standard multinomial RBM, missing ratings are not encoded as zero-valued columns.
Instead, the user-specific RBM only contains visible units for the movies that user has actually rated.
Therefore, if a user has rated \(m\) movies, the visible matrix is \(K \times m\), not \(K \times M\).

**The visible layer in the Netflix-paper RBM is a compact representation of observed ratings only.**

## End-of-notebook summary

1. MovieLens ratings have now been rewritten into the Netflix paper’s visible-unit language.
2. The key object is the user-specific binary indicator matrix.

$$
V \in \{0,1\}^{K \times m}
$$

3. The next notebook will start from this visible matrix.

$$
V
$$

It will then move upward to hidden units and the real-number RBM logic.